# Step 4 — MuJoCo Wipe-Table Evaluation

Full visual benchmark: dataset oracle + ESN + **proximity-gated cloth grasp** (blended attach/release, no teleport) + **wipe task metrics** + 60 FPS video.

Step 3 is **dual-process only** (VLA + ESN latency integration). This notebook replays
`G1_Dex1_Wipe_Table` episode 0 with the same tokens/proprio the ESN was trained on.

```bash
MUJOCO_GL=egl python3 -m src.step4_mujoco_evaluation --episode 0
```

In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("MUJOCO_GL", "egl")

NOTEBOOK_DIR = Path.cwd().resolve()
RESEARCH_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
RESEARCH_DIR = RESEARCH_DIR.resolve()
os.chdir(RESEARCH_DIR)
if str(RESEARCH_DIR) not in sys.path:
    sys.path.insert(0, str(RESEARCH_DIR))

print(f"Research root : {RESEARCH_DIR}")
print(f"Results go to : {RESEARCH_DIR / 'results' / 'step4_mujoco_evaluation'}")

In [ ]:
INIT_EPISODE = 0
DURATION_S = 60.0              # full 1-min video (episode ~12s, loops automatically)
LOOP_EPISODE = True            # repeat episode to fill duration_s
CONTROL_MODE = "kinematic"   # kinematic | pd
CONTROL_HZ = 100.0
VLA_HZ = 2.0
RECORD_VIDEO = True
VIDEO_FPS = 60.0               # 60 fps × 60 s = 3600-frame output
DEVICE = "cuda"
MJCF_PATH = None
ESN_CHECKPOINT = None

print(f"episode={INIT_EPISODE} | duration={DURATION_S:.0f}s | loop={LOOP_EPISODE} | video @ {VIDEO_FPS:.0f} fps")

In [ ]:
import json
import torch

from src.paths import results_path
from src.step3_dual_thread_mujoco import resolve_esn_checkpoint, resolve_mjcf_path
from src.step4_mujoco_evaluation import (
    MuJoCoEvalConfig,
    MuJoCoWipeEvaluator,
    print_eval_summary,
)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA required.")

mjcf = resolve_mjcf_path(MJCF_PATH)
ckpt = resolve_esn_checkpoint(ESN_CHECKPOINT)
out_dir = results_path("step4_mujoco_evaluation")
video_path = out_dir / "table_wipe_benchmark.mp4"

config = MuJoCoEvalConfig(
    mjcf_path=mjcf,
    esn_checkpoint=str(ckpt),
    init_episode=INIT_EPISODE,
    duration_s=DURATION_S,
    control_hz=CONTROL_HZ,
    vla_hz=VLA_HZ,
    control_mode=CONTROL_MODE,
    record_video=RECORD_VIDEO,
    video_path=video_path,
    video_fps=VIDEO_FPS,
    device=DEVICE,
    loop_episode=LOOP_EPISODE,
)

stats = MuJoCoWipeEvaluator(config).run()

report_path = out_dir / "mujoco_eval_report.json"
report = {
    "init_episode": INIT_EPISODE,
    "control_mode": CONTROL_MODE,
    "tracking_rmse": stats.tracking_rmse,
    "grasp_frames": stats.grasp_frames,
    "trajectory_steps": stats.trajectory_steps,
    "video_path": stats.video_path,
    "task_metrics": stats.task_metrics.to_dict() if stats.task_metrics else None,
}
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print_eval_summary(stats, report_path=report_path)

if stats.task_metrics is not None:
    tm = stats.task_metrics
    print(f"\nBenchmark: max_cloth_jump={tm.max_cloth_jump_m:.4f}m | "
          f"wipe_path={tm.wipe_path_length_m:.3f}m | "
          f"grasp_ok={tm.grasp_success}")

In [ ]:
from IPython.display import Video, display

if stats.video_path:
    display(Video(stats.video_path, embed=True, width=640))
else:
    print("No video recorded.")